In [1]:
import pandas as pd

sheets = pd.read_excel("market_data.xlsx", sheet_name=None)
for name, d in sheets.items():
    print(f"Sheet: {name}, Shape: {d.shape}")

Sheet: source_A_industry_assoc, Shape: (37, 5)
Sheet: source_B_research_firm, Shape: (47, 5)
Sheet: drivers, Shape: (48, 4)
Sheet: fx_rates, Shape: (3, 2)


In [2]:
a = sheets["source_A_industry_assoc"]
b = sheets["source_B_research_firm"]
drivers = sheets["drivers"]
fx = sheets["fx_rates"]

In [3]:
a.head(10)

,country,year,market_value,currency,unit
0,Germany,2018,18.2,EUR,million
1,Germany,2019,19.7,EUR,million
2,Germany,2020,18.1,EUR,million
3,Germany,2021,23.6,EUR,million
4,Germany,2022,25.6,EUR,million
5,Germany,2023,28.6,EUR,million
6,Germany,2024,29.9,EUR,million
7,Germany,2025,34.6,EUR,million
8,France,2018,12.6,EUR,million
9,France,2019,13.3,EUR,million


In [4]:
a.dtypes

country             str
year              int64
market_value    float64
currency            str
unit                str
dtype: object

In [5]:
b.dtypes

country             str
year              int64
market_value    float64
currency            str
unit                str
dtype: object

In [6]:
a.groupby(['currency','unit']).size()

currency  unit   
EUR       million    37
dtype: int64

In [7]:
b.groupby(['currency','unit']).size()   

currency  unit   
DKK       million     8
EUR       billion    31
PLN       million     8
dtype: int64

In [8]:
a.groupby(["country","currency","unit"]).size()

country      currency  unit   
Austria      EUR       million    7
France       EUR       million    8
Germany      EUR       million    8
Netherlands  EUR       million    6
Poland       EUR       million    8
dtype: int64

In [9]:
b.groupby(["country","currency","unit"]).size()

country      currency  unit   
Austria      EUR       billion    8
Denmark      DKK       million    8
France       EUR       billion    8
Germany      EUR       billion    7
Netherlands  EUR       billion    8
Poland       PLN       million    8
dtype: int64

In [10]:
#coverage
a.groupby("country")["year"].agg(["count","min","max"])

,count,min,max
country,,,
Austria,7,2018,2025
France,8,2018,2025
Germany,8,2018,2025
Netherlands,6,2018,2025
Poland,8,2018,2025


In [11]:
b.groupby("country")["year"].agg(["count","min","max"])

,count,min,max
country,,,
Austria,8,2018,2025
Denmark,8,2018,2025
France,8,2018,2025
Germany,7,2018,2025
Netherlands,8,2018,2025
Poland,8,2018,2025


In [12]:
a[a["country"] == "France"]

,country,year,market_value,currency,unit
8,France,2018,12.6,EUR,million
9,France,2019,13.3,EUR,million
10,France,2020,12.5,EUR,million
11,France,2021,16.3,EUR,million
12,France,2022,176.0,EUR,million
13,France,2023,20.2,EUR,million
14,France,2024,21.4,EUR,million
15,France,2025,23.6,EUR,million


In [13]:
b[b["country"] == "Austria"]

,country,year,market_value,currency,unit
31,Austria,2018,0.002,EUR,billion
32,Austria,2019,0.002,EUR,billion
33,Austria,2020,0.002,EUR,billion
34,Austria,2021,0.003,EUR,billion
35,Austria,2022,0.003,EUR,billion
36,Austria,2023,0.003,EUR,billion
37,Austria,2024,0.003,EUR,billion
38,Austria,2025,0.004,EUR,billion


Standardise units and currency

In [14]:
fx

,currency,eur_per_unit
0,EUR,1.000000
1,PLN,0.232558
2,DKK,0.134228


In [15]:
UNIT_FACTOR = {"million": 1, "billion": 1000}

def standarsize(df):
    out = df.merge(fx, on="currency", how="left")
    out["unit_factor"] = out["unit"].map(UNIT_FACTOR)
    out["value_eur_m"] = out["market_value"] * out["unit_factor"] * out["eur_per_unit"]
    return out

a_std = standarsize(a)
b_std = standarsize(b)

In [16]:
b_std.head(10)

,country,year,market_value,currency,unit,eur_per_unit,unit_factor,value_eur_m
0,Germany,2018,0.021,EUR,billion,1.0,1000,21.0
1,Germany,2020,0.021,EUR,billion,1.0,1000,21.0
2,Germany,2021,0.029,EUR,billion,1.0,1000,29.0
3,Germany,2022,0.031,EUR,billion,1.0,1000,31.0
4,Germany,2023,0.033,EUR,billion,1.0,1000,33.0
5,Germany,2024,0.036,EUR,billion,1.0,1000,36.0
6,Germany,2025,0.041,EUR,billion,1.0,1000,41.0
7,France,2018,0.014,EUR,billion,1.0,1000,14.0
8,France,2019,0.016,EUR,billion,1.0,1000,16.0
9,France,2020,0.015,EUR,billion,1.0,1000,15.0


In [17]:
a_std["value_eur_m"].isna().sum()

np.int64(0)

In [18]:
b_std["value_eur_m"].isna().sum()

np.int64(0)

In [19]:
#compare and join
compare = a_std[["country", "year", "value_eur_m"]].merge(
    b_std[["country", "year", "value_eur_m"]],
    on=["country", "year"],
    suffixes=("_a", "_b"),
)
compare["ratio_b_a"] = compare["value_eur_m_b"] / compare["value_eur_m_a"]
compare.round(2)

,country,year,value_eur_m_a,value_eur_m_b,ratio_b_a
0,Germany,2018,18.2,21.00,1.15
1,Germany,2020,18.1,21.00,1.16
2,Germany,2021,23.6,29.00,1.23
3,Germany,2022,25.6,31.00,1.21
4,Germany,2023,28.6,33.00,1.15
5,Germany,2024,29.9,36.00,1.20
6,Germany,2025,34.6,41.00,1.18
7,France,2018,12.6,14.00,1.11
8,France,2019,13.3,16.00,1.20
9,France,2020,12.5,15.00,1.20


In [20]:
#quality checks
countries = sorted(set(a_std["country"]) | set(b_std["country"]))
years = list(range(2018, 2026))

grid = pd.MultiIndex.from_product(
    [countries, years], names=["country", "year"]
).to_frame(index=False)

In [21]:
#completeness check
qa = grid.merge(a_std[["country", "year", "value_eur_m"]],
                on=["country", "year"],
                how="left")
qa["flag_missing"] = qa["value_eur_m"].isna()
qa["flag_missing"].sum()

np.int64(11)

In [22]:
#plausibility check (YoY growth)
qa = qa.sort_values(["country", "year"])
qa["yoy"] = qa.groupby("country")["value_eur_m"].pct_change(fill_method=None)
qa["flag_yoy_jump"] = qa["yoy"].abs() > 0.5

In [23]:
#consistency check (source A vs source B)
qa = qa.merge(b_std[["country", "year", "value_eur_m"]].rename(columns={"value_eur_m": "value_b"}),
              on=["country", "year"], how="left")
qa["ratio_b_to_a"] = qa["value_b"] / qa["value_eur_m"]

typical_ratio = qa["ratio_b_to_a"].median()
qa["flag_source_mismatch"] = (qa["ratio_b_to_a"] / typical_ratio - 1).abs() > 0.5

print("Typical ratio:", round(typical_ratio, 2))
qa[qa["flag_source_mismatch"]]

Typical ratio: 1.19


,country,year,value_eur_m,flag_missing,yoy,flag_yoy_jump,value_b,ratio_b_to_a,flag_source_mismatch
20,France,2022,176.0,False,9.797546,True,22.0,0.125,True


In [24]:
qa

,country,year,value_eur_m,flag_missing,yoy,flag_yoy_jump,value_b,ratio_b_to_a,flag_source_mismatch
0,Austria,2018,2.0,False,NaN,False,2.000000,1.000000,False
1,Austria,2019,2.2,False,0.100000,False,2.000000,0.909091,False
2,Austria,2020,1.9,False,-0.136364,False,2.000000,1.052632,False
3,Austria,2021,2.5,False,0.315789,False,3.000000,1.200000,False
4,Austria,2022,2.6,False,0.040000,False,3.000000,1.153846,False
5,Austria,2023,NaN,True,NaN,False,3.000000,NaN,False
6,Austria,2024,3.0,False,NaN,False,3.000000,1.000000,False
7,Austria,2025,3.3,False,0.100000,False,4.000000,1.212121,False
8,Denmark,2018,NaN,True,NaN,False,1.879195,NaN,False
9,Denmark,2019,NaN,True,NaN,False,2.013423,NaN,False


In [25]:
flag_cols = ["flag_missing", "flag_yoy_jump", "flag_source_mismatch"]
qa[flag_cols].sum()

flag_missing            11
flag_yoy_jump            2
flag_source_mismatch     1
dtype: int64

In [26]:
qa[qa[flag_cols].any(axis=1)]

,country,year,value_eur_m,flag_missing,yoy,flag_yoy_jump,value_b,ratio_b_to_a,flag_source_mismatch
5,Austria,2023,NaN,True,NaN,False,3.000000,NaN,False
8,Denmark,2018,NaN,True,NaN,False,1.879195,NaN,False
9,Denmark,2019,NaN,True,NaN,False,2.013423,NaN,False
10,Denmark,2020,NaN,True,NaN,False,2.013423,NaN,False
11,Denmark,2021,NaN,True,NaN,False,2.281879,NaN,False
12,Denmark,2022,NaN,True,NaN,False,2.684564,NaN,False
13,Denmark,2023,NaN,True,NaN,False,2.953020,NaN,False
14,Denmark,2024,NaN,True,NaN,False,2.953020,NaN,False
15,Denmark,2025,NaN,True,NaN,False,3.624161,NaN,False
20,France,2022,176.0,False,9.797546,True,22.000000,0.125000,True


In [27]:
qa

,country,year,value_eur_m,flag_missing,yoy,flag_yoy_jump,value_b,ratio_b_to_a,flag_source_mismatch
0,Austria,2018,2.0,False,NaN,False,2.000000,1.000000,False
1,Austria,2019,2.2,False,0.100000,False,2.000000,0.909091,False
2,Austria,2020,1.9,False,-0.136364,False,2.000000,1.052632,False
3,Austria,2021,2.5,False,0.315789,False,3.000000,1.200000,False
4,Austria,2022,2.6,False,0.040000,False,3.000000,1.153846,False
5,Austria,2023,NaN,True,NaN,False,3.000000,NaN,False
6,Austria,2024,3.0,False,NaN,False,3.000000,1.000000,False
7,Austria,2025,3.3,False,0.100000,False,4.000000,1.212121,False
8,Denmark,2018,NaN,True,NaN,False,1.879195,NaN,False
9,Denmark,2019,NaN,True,NaN,False,2.013423,NaN,False


In [28]:
#Reconcile two sources
m = qa[["country", "year", "value_eur_m", "value_b", "flag_source_mismatch"]].copy()
m = m.rename(columns={"value_eur_m": "value_a"})

m["flag_invalid_a"] = m["flag_source_mismatch"]
m.loc[m["flag_invalid_a"], "value_a"] = float("nan")


In [29]:
m[m["country"] == "France"]

,country,year,value_a,value_b,flag_source_mismatch,flag_invalid_a
16,France,2018,12.6,14.0,False,False
17,France,2019,13.3,16.0,False,False
18,France,2020,12.5,15.0,False,False
19,France,2021,16.3,20.0,False,False
20,France,2022,NaN,22.0,True,True
21,France,2023,20.2,23.0,False,False
22,France,2024,21.4,24.0,False,False
23,France,2025,23.6,28.0,False,False


In [30]:
#measure scaling factor per country
scales = m.dropna(subset=["value_a","value_b"]).copy()
scales["ratio"] = scales["value_b"] / scales["value_a"]

factor_table = scales.groupby("country")["ratio"].agg(["median","std","count"]).round(3)
factor_table

,median,std,count
country,,,
Austria,1.053,0.115,7
France,1.186,0.045,7
Germany,1.185,0.030,7
Netherlands,1.204,0.047,6
Poland,1.196,0.027,8


In [31]:
scales 

,country,year,value_a,value_b,flag_source_mismatch,flag_invalid_a,ratio
0,Austria,2018,2.0,2.000000,False,False,1.000000
1,Austria,2019,2.2,2.000000,False,False,0.909091
2,Austria,2020,1.9,2.000000,False,False,1.052632
3,Austria,2021,2.5,3.000000,False,False,1.200000
4,Austria,2022,2.6,3.000000,False,False,1.153846
6,Austria,2024,3.0,3.000000,False,False,1.000000
7,Austria,2025,3.3,4.000000,False,False,1.212121
16,France,2018,12.6,14.000000,False,False,1.111111
17,France,2019,13.3,16.000000,False,False,1.203008
18,France,2020,12.5,15.000000,False,False,1.200000


In [32]:
#decide factor for each country
global_factor = scales["ratio"].median()
print(round(global_factor, 3))

1.19


In [33]:
country_factor = scales.groupby("country")["ratio"].median()

In [34]:
country_factor 

country
Austria        1.052632
France         1.186441
Germany        1.184971
Netherlands    1.203887
Poland         1.195726
Name: ratio, dtype: float64

In [35]:
country_spread = scales.groupby("country")["ratio"].std()

In [36]:
country_spread

country
Austria        0.115412
France         0.045301
Germany        0.030300
Netherlands    0.047358
Poland         0.027229
Name: ratio, dtype: float64

In [37]:
MAX_SPREAD = 0.05

def pick_factor(country):
    if country in country_spread and country_spread[country] < MAX_SPREAD:
        return country_factor[country]
    return global_factor


In [38]:
m["factor"] = m["country"].map(pick_factor)
m.groupby("country")["factor"].first().round(3)

country
Austria        1.190
Denmark        1.190
France         1.186
Germany        1.185
Netherlands    1.204
Poland         1.196
Name: factor, dtype: float64

In [39]:
m

,country,year,value_a,value_b,flag_source_mismatch,flag_invalid_a,factor
0,Austria,2018,2.0,2.000000,False,False,1.190476
1,Austria,2019,2.2,2.000000,False,False,1.190476
2,Austria,2020,1.9,2.000000,False,False,1.190476
3,Austria,2021,2.5,3.000000,False,False,1.190476
4,Austria,2022,2.6,3.000000,False,False,1.190476
5,Austria,2023,NaN,3.000000,False,False,1.190476
6,Austria,2024,3.0,3.000000,False,False,1.190476
7,Austria,2025,3.3,4.000000,False,False,1.190476
8,Denmark,2018,NaN,1.879195,False,False,1.190476
9,Denmark,2019,NaN,2.013423,False,False,1.190476


In [41]:
m["value_b_scaled"] = m["value_b"] / m["factor"]

m["value_final"] = m["value_a"].fillna(m["value_b_scaled"])

m["value_origin"] = "A"
m.loc[m["value_a"].isna() & m["value_b"].notna(), "value_origin"] = "B_scaled"
m.loc[m["value_final"].isna(), "value_origin"] = "missing"

print(m["value_origin"].value_counts())
m[m["value_origin"] != "A"][["country", "year", "value_b", "factor", "value_final", "value_origin"]].round(2)

value_origin
A           36
B_scaled    12
Name: count, dtype: int64


,country,year,value_b,factor,value_final,value_origin
5,Austria,2023,3.00,1.19,2.52,B_scaled
8,Denmark,2018,1.88,1.19,1.58,B_scaled
9,Denmark,2019,2.01,1.19,1.69,B_scaled
10,Denmark,2020,2.01,1.19,1.69,B_scaled
11,Denmark,2021,2.28,1.19,1.92,B_scaled
12,Denmark,2022,2.68,1.19,2.26,B_scaled
13,Denmark,2023,2.95,1.19,2.48,B_scaled
14,Denmark,2024,2.95,1.19,2.48,B_scaled
15,Denmark,2025,3.62,1.19,3.04,B_scaled
20,France,2022,22.00,1.19,18.54,B_scaled


In [42]:
m[m["country"].isin(["France", "Netherlands"])][["country", "year", "value_final", "value_origin"]].round(2)

,country,year,value_final,value_origin
16,France,2018,12.60,A
17,France,2019,13.30,A
18,France,2020,12.50,A
19,France,2021,16.30,A
20,France,2022,18.54,B_scaled
21,France,2023,20.20,A
22,France,2024,21.40,A
23,France,2025,23.60,A
32,Netherlands,2018,4.50,A
33,Netherlands,2019,4.90,A


To test which gap-filling method is best

In [44]:
drivers

,country,year,population_m,gdp_trillion_eur
0,Germany,2018,83.30,4.500
1,Germany,2019,83.47,4.635
2,Germany,2020,83.63,4.532
3,Germany,2021,83.80,4.905
4,Germany,2022,83.97,5.040
5,Germany,2023,84.13,5.175
6,Germany,2024,84.30,5.310
7,Germany,2025,84.47,5.445
8,France,2018,68.20,3.100
9,France,2019,68.34,3.193


In [45]:
test = m[["country","year","value_a","value_b_scaled"]].merge(drivers, on=["country","year"], how="left")
test = test.sort_values(["country","year"])
g = test.groupby("country")

In [53]:
#method 1: interpolation = avg of prev and next year
test["est_interp"] = (g["value_a"].shift(1) + g["value_a"].shift(-1))/2

#method 2: driver based = interpolate the share of gdp, multiply by this years gdp
test["share"] = test["value_a"] / test["gdp_trillion_eur"]
test["est_driver"] = (g["share"].shift(1) + g["share"].shift(-1)) / 2 *test["gdp_trillion_eur"]

#method 3: source B scaled
test["est_source_b"] = test["value_b_scaled"]

In [55]:
t = test.dropna(subset=["value_a","est_interp","est_driver", "est_source_b"]).copy()

for method in ["interp", "driver", "source_b"]:
    t[f"ape_{method}"] = (t[f"est_{method}"] / t["value_a"] - 1).abs()

ape_cols = ["ape_interp", "ape_driver", "ape_source_b"]

In [56]:
len(t)

19

In [57]:
t[ape_cols].mean().round(3)

ape_interp      0.078
ape_driver      0.059
ape_source_b    0.038
dtype: float64

In [58]:
t.groupby("year")[ape_cols].mean().round(3)

,ape_interp,ape_driver,ape_source_b
year,,,
2019,0.081,0.057,0.093
2020,0.195,0.133,0.038
2021,0.076,0.058,0.023
2022,0.016,0.015,0.013
2023,0.024,0.024,0.023
2024,0.034,0.034,0.030


In [59]:
t.groupby("country")[ape_cols].mean().round(3)

,ape_interp,ape_driver,ape_source_b
country,,,
Austria,0.150,0.115,0.120
France,0.088,0.059,0.027
Germany,0.075,0.059,0.024
Netherlands,0.033,0.033,0.025
Poland,0.055,0.039,0.017


Key Performance Indicators (how fast is the market growing?)

In [61]:
market = (
    m[["country","year","value_final", "value_origin"]]
    .rename(columns={"value_final": "market_eur_m"})
    .merge(drivers, on=["country", "year"], how="left")
    .sort_values(["country","year"])
    .reset_index(drop=True)
)

In [63]:
assert len(market) == 48
assert market["market_eur_m"].notna().all()

market.to_csv("market_clean.csv", index=False)

In [65]:
market.head(10)

,country,year,market_eur_m,value_origin,population_m,gdp_trillion_eur
0,Austria,2018,2.000000,A,9.10,0.500
1,Austria,2019,2.200000,A,9.12,0.515
2,Austria,2020,1.900000,A,9.14,0.503
3,Austria,2021,2.500000,A,9.15,0.545
4,Austria,2022,2.600000,A,9.17,0.560
5,Austria,2023,2.520000,B_scaled,9.19,0.575
6,Austria,2024,3.000000,A,9.21,0.590
7,Austria,2025,3.300000,A,9.23,0.605
8,Denmark,2018,1.578523,B_scaled,5.90,0.400
9,Denmark,2019,1.691275,B_scaled,5.91,0.412


In [66]:
#YoY growth
market["yoy"] = market.groupby("country")["market_eur_m"].pct_change()

market.pivot(index="year", columns="country", values="yoy").round(3)

country,Austria,Denmark,France,Germany,Netherlands,Poland
year,,,,,,
2018,NaN,NaN,NaN,NaN,NaN,NaN
2019,0.100,0.071,0.056,0.082,0.089,0.062
2020,-0.136,0.000,-0.060,-0.081,-0.152,-0.088
2021,0.316,0.133,0.304,0.304,0.400,0.226
2022,0.040,0.176,0.138,0.085,0.135,0.079
2023,-0.031,0.100,0.089,0.117,0.091,0.049
2024,0.190,0.000,0.059,0.045,0.167,0.047
2025,0.100,0.227,0.103,0.157,0.095,0.111


In [67]:
#compound annual growth rate
start = market[market["year"] == 2018].set_index("country")["market_eur_m"]
end = market[market["year"] == 2025].set_index("country")["market_eur_m"]


In [70]:
cagr = (end / start) ** (1 / 7) - 1
cagr.sort_values(ascending=False).round(3)

country
Netherlands    0.108
Denmark        0.098
Germany        0.096
France         0.094
Austria        0.074
Poland         0.066
Name: market_eur_m, dtype: float64

In [72]:
#per capita
market["per_capita_eur"] = market["market_eur_m"] / market["population_m"]

market[market["year"] == 2025][["country", "per_capita_eur"]].sort_values("per_capita_eur", ascending=False).round(2)

,country,per_capita_eur
15,Denmark,0.51
39,Netherlands,0.51
31,Germany,0.41
7,Austria,0.36
23,France,0.34
47,Poland,0.13


In [73]:
#market share
market["share"] = market["market_eur_m"] / market.groupby("year")["market_eur_m"].transform("sum")

market[market["year"] == 2025][["country", "market_eur_m", "share"]].sort_values("share", ascending=False).round(3)

,country,market_eur_m,share
31,Germany,34.600,0.439
23,France,23.600,0.300
39,Netherlands,9.200,0.117
47,Poland,5.000,0.063
7,Austria,3.300,0.042
15,Denmark,3.044,0.039


In [75]:
market.loc[market["country"] == "Denmark", ["year", "population_m"]]

,year,population_m
8,2018,5.90
9,2019,5.91
10,2020,5.92
11,2021,5.94
12,2022,5.95
13,2023,5.96
14,2024,5.97
15,2025,5.98


Forecasting (build->test->forecast)

In [77]:
import numpy as np
import warnings
from statsmodels.tsa.holtwinters import ExponentialSmoothing

warnings.filterwarnings("ignore")

In [78]:
mk = pd.read_csv("market_clean.csv")
TRAIN_END = 2022

In [ ]:
rows = []
for country, g in mk.groupby("country"):
    g = g.sort_values("year")
    train = g[g["year"] <= TRAIN_END]["market_eur_m"].values
    test = g[g["year"] > TRAIN_END]

    last = train[-1]
    cagr = (train[-1] / train[0]) ** (1 / (len(train) - 1)) - 1
    holt = ExponentialSmoothing(train, trend="add").fit().forecast(len(test))

    for i, (year, actual) in enumerate(zip(test["year"], test["market_eur_m"])):
        rows.append({
            "country": country,
            "year": year,
            "actual": actual,
            "naive": last,
            "cagr": last * (1+cagr) ** (i + 1),
            "holt": holt[i]
        })

bt = pd.DataFrame(rows)
for method in ["naive", "cagr", "holt"]:
    bt[f"ape_{method}"] = (bt[method] / bt["actual"] - 1).abs()



In [82]:
ape_cols = ["ape_naive", "ape_cagr", "ape_holt"]
bt[ape_cols].mean().round(3)

ape_naive    0.147
ape_cagr     0.035
ape_holt     0.070
dtype: float64

In [83]:
bt.groupby("country")[ape_cols].mean().round(3)

,ape_naive,ape_cagr,ape_holt
country,,,
Austria,0.126,0.051,0.072
Denmark,0.147,0.042,0.054
France,0.143,0.037,0.052
Germany,0.170,0.028,0.079
Netherlands,0.193,0.034,0.120
Poland,0.105,0.019,0.042


In [90]:
value_2018 = mk[mk["year"] == 2018].set_index("country")["market_eur_m"]
value_2025 = mk[mk["year"] == 2025].set_index("country")["market_eur_m"]

In [93]:
cagr_full = (value_2025 / value_2018) ** (1/7) -1

In [95]:
forecast = pd.DataFrame([
    {"country": c, "year": y, "forecast_eur_m": value_2025[c] * (1 + cagr_full[c]) ** (y - 2025)}
    for c in cagr.index
    for y in range(2026, 2031)
])

In [96]:
forecast[forecast["year"] == 2030].round(1)

,country,year,forecast_eur_m
4,Austria,2030,4.7
9,Denmark,2030,4.9
14,France,2030,36.9
19,Germany,2030,54.7
24,Netherlands,2030,15.3
29,Poland,2030,6.9


In [97]:
forecast.groupby("year")["forecast_eur_m"].sum().round(1)

year
2026     86.1
2027     94.3
2028    103.1
2029    112.8
2030    123.5
Name: forecast_eur_m, dtype: float64

In [98]:
forecast.to_csv("market_forecast", index=False)